In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import os
OUTPUT_DIR = "Code Outputs/Gap Interpolation Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUTPUT_DIR, n)
    
# Configuration
INPUT_FILE = "Code Outputs/Fusion Outputs/Unified_BiasAligned_Levels.xlsx"
VALUE_COL  = "Unified_Level_m"
MAX_GAP    = None   # None = fill all internal gaps

# Loading data
uni = pd.read_excel(INPUT_FILE)
uni["Date"] = pd.to_datetime(uni["Date"])

def run_lengths(is_missing):

    out = np.zeros(len(is_missing), dtype=int)
    i = 0
    arr = is_missing.values
    while i < len(arr):
        if arr[i]:
            j = i
            while j < len(arr) and arr[j]:
                j += 1
            out[i:j] = j - i
            i = j
        else:
            i += 1
    return out

filled_frames = []
cov_rows = []

for lake, g in uni.groupby("Reservoir"):
    g = g.sort_values("Date").set_index("Date")
    # ensure a continuous monthly index
    full = pd.date_range(g.index.min(), g.index.max(), freq="MS")
    g = g.reindex(full)

    series  = g[VALUE_COL].copy()
    missing = series.isna()

    # only internal gaps are candidates
    first, last = series.first_valid_index(), series.last_valid_index()
    internal = missing.copy()
    internal[:first] = False
    internal[last:]  = False

    # optionally exclude runs longer than MAX_GAP
    if MAX_GAP is not None:
        lengths = run_lengths(missing)
        too_long = pd.Series(lengths > MAX_GAP, index=series.index)
        internal = internal & ~too_long

    # time-interpolate
    interp_full = series.interpolate(method="time", limit_area="inside")
    filled = series.copy()
    filled[internal] = interp_full[internal]

    flag = pd.Series(False, index=series.index)
    flag[internal & filled.notna()] = True

    df_out = pd.DataFrame({
        "Date": series.index,
        "Reservoir": lake,
        "Level_m": filled.values,            # observed where available, else interpolated
        "is_interpolated": flag.values,
        "Source": g["Source"].values,        # original provenance (NaN where interpolated)
    })
    filled_frames.append(df_out)

    # coverage stats
    n_total = filled.notna().sum()
    n_interp = int(flag.sum())
    n_obs = n_total - n_interp
    longest = int(run_lengths(missing & internal).max()) if internal.any() else 0
    still_missing = int(filled.isna().sum())   # only at the ends, or > MAX_GAP runs
    cov_rows.append({
        "Lake": lake,
        "Span_months": len(full),
        "Observed": int(n_obs),
        "Interpolated": n_interp,
        "Pct_interpolated": round(100 * n_interp / len(full), 1),
        "Longest_interp_run_m": longest,
        "Still_missing": still_missing,
    })

result = pd.concat(filled_frames, ignore_index=True)
coverage = pd.DataFrame(cov_rows)

result.to_excel(out("Unified_Interpolated_Levels.xlsx"), index=False)
coverage.to_csv(out("Coverage_Table.csv"), index=False)

print("=== COVERAGE TABLE ===")
print(coverage.to_string(index=False))

# Figure 2, observed vs interpolated

lakes = list(result["Reservoir"].unique())
fig, axes = plt.subplots(4, 2, figsize=(15, 16)); axes = axes.flatten()
for i, lake in enumerate(lakes):
    g = result[result["Reservoir"] == lake]; ax = axes[i]
    ax.plot(g["Date"], g["Level_m"], "-", lw=.8, color="lightsteelblue", zorder=1)
    obs = g[~g["is_interpolated"]]
    ip  = g[g["is_interpolated"]]
    ax.plot(obs["Date"], obs["Level_m"], ".", ms=3, color="tab:blue", label="observed")
    ax.plot(ip["Date"],  ip["Level_m"],  ".", ms=5, color="red",      label="interpolated")
    ax.set_title(lake, fontsize=11, fontweight="bold"); ax.set_ylabel("Level (m)")
    if i == 0:
        ax.legend(fontsize=8)
axes[7].set_visible(False)
plt.suptitle("Observed vs Interpolated Monthly Levels", fontsize=14, fontweight="bold", y=.995)
plt.tight_layout(); plt.savefig(out("Coverage_Diag.png"), dpi=800); plt.close()

=== COVERAGE TABLE ===
           Lake  Span_months  Observed  Interpolated  Pct_interpolated  Longest_interp_run_m  Still_missing
    Lake Albert          370       355            15               4.1                     1              0
    Lake Edward          371       330            41              11.1                     6              0
      Lake Kivu          371       317            54              14.6                     4              0
    Lake Malawi          403       403             0               0.0                     0              0
Lake Tanganyika          401       401             0               0.0                     0              0
   Lake Turkana          402       401             1               0.2                     1              0
  Lake Victoria          402       402             0               0.0                     0              0
